In [448]:
import numpy as np 
import pandas as pd 
from matplotlib import pyplot as plt
import seaborn as sns 


In [449]:
## CPLEX reading results 
# read data from cplex: read "LB": 55, objective, and solution given in this format ("solution": [
#        "id32",
#        "id36",
#        "id37",
#        "id40",
#        "id49",
#        "id57",
#        "id67",)
#Each instance/file information is given per each row, its properties are extracted from the filename 
# (e.g., embeddings_large_GPQA_CHUNKED.jsonl_nemotron_correlation_p10_alg0.json) is mapped to ==> mapped to (large, GPQA_CHUNKED, nemotron, correlation, p10, alg0) as columns (size, dataset, model_type, measure, threshold, alg)
#Read json in the DIR, extract the information from there, and pack it into Dataframe as explained there:



In [450]:
import os
import json
import glob
import pandas as pd


############################################################
# Filename parser (handles both formats)
############################################################

def parse_filename(filename):

    base = os.path.basename(filename)
    base = os.path.splitext(base)[0]   # removes ONLY .json

    parts = base.split("_")

    ########################################################
    # FORMAT 1:
    # embeddings_large_GPQA_CHUNKED.jsonl_nemotron_...
    ########################################################
    if parts[0] == "embeddings":

        size = parts[1]

        jsonl_index = None
        for i, p in enumerate(parts):
            if ".jsonl" in p:
                jsonl_index = i
                break

        if jsonl_index is None:
            raise ValueError(f"No .jsonl token found in {filename}")

        dataset = "_".join(parts[2:jsonl_index + 1]).replace(".jsonl", "")

        model_type = parts[jsonl_index + 1]
        measure = parts[jsonl_index + 2]
        threshold = parts[jsonl_index + 3]
        alg = parts[jsonl_index + 4]

    ########################################################
    # FORMAT 2:
    # small_embeds_sp_GPQA_fixed.jsonl_e5_cosine_p10_alg0
    ########################################################
    elif parts[0] == "small":

        size = "small"

        jsonl_index = None
        for i, p in enumerate(parts):
            if ".jsonl" in p:
                jsonl_index = i
                break

        if jsonl_index is None:
            raise ValueError(f"No .jsonl token found in {filename}")

        dataset = "_".join(parts[3:jsonl_index + 1])
        dataset = dataset.replace("_fixed.jsonl", "").replace(".jsonl", "")

        model_type = parts[jsonl_index + 1]
        measure = parts[jsonl_index + 2]
        threshold = parts[jsonl_index + 3]
        alg = parts[jsonl_index + 4]

    else:
        raise ValueError(f"Unknown filename format: {filename}")

    return {
        "size": size,
        "dataset": dataset,
        "model_type": model_type,
        "measure": measure,
        "threshold": threshold,
        "alg": alg
    }


############################################################
# Parse all CPLEX solution JSON files
############################################################

def cplex_parse_all(DIR):

    rows = []

    files = glob.glob(os.path.join(DIR, "*.json"))
    print("Found files:", len(files))

    for file in files:

        #print("Reading:", file)

        try:

            ####################################################
            # Parse metadata from filename
            ####################################################
            meta = parse_filename(file)

            ####################################################
            # Load JSON result
            ####################################################
            with open(file, "r") as f:
                data = json.load(f)

            ####################################################
            # Extract solver outputs
            ####################################################
            LB = data.get("LB", None)
            objective = data.get("objective", None)
            status = data.get("status", None)
            runtime = data.get("time", None)
            solution = data.get("solution", [])

            ####################################################
            # Build row
            ####################################################
            row = {
                "file": os.path.basename(file),

                "size": meta["size"],
                "dataset": meta["dataset"],
                "model_type": meta["model_type"],
                "measure": meta["measure"],
                "threshold": meta["threshold"],
                "alg": meta["alg"],

                "LB": LB,
                "objective": objective,
                "status": status,
                "runtime": runtime,

                "solution_size": len(solution),
                "solution": solution
            }

            rows.append(row)

        except Exception as e:
            print("ERROR:", file)
            print(e)

    return pd.DataFrame(rows)


cplex_data = cplex_parse_all("cplex")
cplex_data = cplex_data.drop(columns=["file"], axis=1)

Found files: 1186


In [451]:
cplex_data = cplex_data[["dataset", "model_type", "size", "measure", "threshold", "objective", "alg", "LB", "solution"]]
cplex_data
cplex_data 

,dataset,model_type,size,measure,threshold,objective,alg,LB,solution
0,GPQA_CHUNKED,qwen8b,large,correlation,p80,318.0,alg0,318,"[id0, id1, id2, id4, id6, id7, id10, id11, id1..."
1,GPQA,e5,small,seuclidean,p10,1.0,alg1,1,[id17]
2,IFEval_NOCHUNK,qwen8b,large,seuclidean,p20,105.0,alg0,105,"[id1094, id1127, id1147, id1187, id1262, id126..."
3,IFEval_NOCHUNK,qwen4b,large,cosine,p50,180.0,alg0,180,"[id1019, id1040, id1082, id1094, id1127, id112..."
4,GPQA_CHUNKED,nemotron,large,correlation,p80,1.0,alg1,1,[id359]
...,...,...,...,...,...,...,...,...,...
1181,IFEval_CHUNKED,qwen8b,large,correlation,p20,1.0,alg1,1,[id1000]
1182,Omni-MATH,e5,small,seuclidean,p20,1.0,alg1,1,[id124]
1183,IFEval_CHUNKED,qwen4b,large,cosine,p80,344.0,alg0,344,"[id1001, id1019, id102, id1040, id1069, id1072..."
1184,GPQA,bge,small,cosine,p10,71.0,alg0,71,"[id2, id3, id6, id28, id34, id36, id37, id40, ..."


In [452]:
## GREEDY parse results:
## read greedy results as well: appart from the information given by file name, you have to read here the following

############################################################
# Parse GREEDY heuristic solution files
############################################################

import os
import json
import glob
import pandas as pd


############################################################
# Parse GREEDY heuristic solution files
############################################################

def greedy_parse_all(DIR):

    rows = []

    ########################################################
    # Read all json files
    ########################################################
    for file_path in glob.glob(os.path.join(DIR, "*.json")):

        filename = os.path.basename(file_path)

        try:

            ################################################
            # Remove ".json"
            ################################################
            name = filename[:-5]

            ################################################
            # Split by "_"
            ################################################
            parts = name.split("_")

            ################################################
            # CASE 1:
            #
            # embeddings_large_GPQA_CHUNKED.jsonl_nemotron...
            #
            ################################################
            if parts[0] == "embeddings":

                ################################################
                # size
                ################################################
                size = parts[1]

                ################################################
                # Find token containing ".jsonl"
                ################################################
                jsonl_index = None

                for i, p in enumerate(parts):

                    if ".jsonl" in p:
                        jsonl_index = i
                        break

                if jsonl_index is None:
                    raise ValueError("No .jsonl token found")

                ################################################
                # Dataset
                ################################################
                dataset_parts = parts[2:jsonl_index + 1]

                dataset = "_".join(dataset_parts)

                dataset = dataset.replace(".jsonl", "")

                ################################################
                # Remaining fields
                ################################################
                model_type = parts[jsonl_index + 1]

                measure = parts[jsonl_index + 2]

                threshold = parts[jsonl_index + 3]

                alg = parts[jsonl_index + 4]

            ################################################
            # CASE 2:
            #
            # small_embeds_sp_GPQA_fixed.jsonl_e5...
            #
            ################################################
            elif parts[0] == "small":

                ################################################
                # size
                ################################################
                size = "small"

                ################################################
                # Find token containing ".jsonl"
                ################################################
                jsonl_index = None

                for i, p in enumerate(parts):

                    if ".jsonl" in p:
                        jsonl_index = i
                        break

                if jsonl_index is None:
                    raise ValueError("No .jsonl token found")

                ################################################
                # Dataset extraction
                #
                # small_embeds_sp_GPQA_fixed.jsonl
                #
                ################################################
                dataset_parts = parts[3:jsonl_index + 1]

                dataset = "_".join(dataset_parts)

                dataset = dataset.replace("_fixed.jsonl", "")
                dataset = dataset.replace(".jsonl", "")

                ################################################
                # Remaining fields
                ################################################
                model_type = parts[jsonl_index + 1]

                measure = parts[jsonl_index + 2]

                threshold = parts[jsonl_index + 3]

                alg = parts[jsonl_index + 4]

            else:

                raise ValueError("Unknown filename pattern")

            ################################################
            # Load JSON
            ################################################
            with open(file_path, "r") as f:

                data = json.load(f)

            ################################################
            # Extract info
            ################################################
            objective = data.get("objective", None)

            runtime = data.get("time", None)

            ################################################
            # Greedy files use "nodes"
            ################################################
            solution = data.get("nodes", [])

            solution_size = len(solution)

            ################################################
            # Store row
            ################################################
            row = {
                "file": filename,
                "size": size,
                "dataset": dataset,
                "model_type": model_type,
                "measure": measure,
                "threshold": threshold,
                "alg": alg,
                "objective": objective,
                "solution_size": solution_size,
                "time": runtime,
                "solution": solution
            }

            rows.append(row)

        except Exception as e:

            print(f"ERROR parsing {filename}: {e}")

    ########################################################
    # Create dataframe
    ########################################################
    df = pd.DataFrame(rows)

    return df

greedy_data = greedy_parse_all("greedy")
greedy_data = greedy_data.drop(columns=["file"])


In [453]:
greedy_data = greedy_data[["dataset", "model_type", "size", "measure", "threshold", "alg", "objective", "solution"]]
greedy_data #[["objective"]].mean()

,dataset,model_type,size,measure,threshold,alg,objective,solution
0,GPQA_CHUNKED,qwen8b,large,correlation,p80,alg0,318,"[id127, id350, id175, id123, id69, id370, id11..."
1,GPQA,e5,small,seuclidean,p10,alg1,1,[id17]
2,IFEval_NOCHUNK,qwen8b,large,seuclidean,p20,alg0,105,"[id2505, id1127, id3048, id2350, id2871, id153..."
3,IFEval_NOCHUNK,qwen4b,large,cosine,p50,alg0,180,"[id3069, id1561, id2605, id340, id3739, id1601..."
4,GPQA_CHUNKED,nemotron,large,correlation,p80,alg1,1,[id359]
...,...,...,...,...,...,...,...,...
1183,IFEval_CHUNKED,qwen8b,large,correlation,p20,alg1,1,[id1000]
1184,Omni-MATH,e5,small,seuclidean,p20,alg1,1,[id3884]
1185,IFEval_CHUNKED,qwen4b,large,cosine,p80,alg0,344,"[id3606, id1546, id1561, id3672, id2239, id128..."
1186,GPQA,bge,small,cosine,p10,alg0,71,"[id418, id308, id67, id357, id176, id387, id19..."


In [454]:
## numds DATA: 



In [455]:
import os
import glob
import re
import pandas as pd


# =========================================================
# Parse filename metadata (ROBUST)
# =========================================================
def parse_numds_filename(filename):

    base = os.path.basename(filename)
    base = base.replace(".out", "")

    ########################################################
    # seed + time extraction
    ########################################################
    seed = None
    time_limit = None

    m_seed = re.search(r"seed(\d+)", base)
    if m_seed:
        seed = int(m_seed.group(1))

    m_time = re.search(r"t(\d+)", base)
    if m_time:
        time_limit = int(m_time.group(1))

    ########################################################
    # remove suffix after .dimacs
    ########################################################
    if ".dimacs" in base:
        core = base.split(".dimacs")[0]
    else:
        core = base

    tokens = core.split("_")

    ########################################################
    # SIZE
    ########################################################
    size = "unknown"
    if "large" in tokens:
        size = "large"
    elif "small" in tokens[0]:
        size = "small"

    ########################################################
    # DATASET (preserves CHUNKED / NOCHUNK)
    ########################################################
    dataset = None

    '''
    if "GPQA_CHUNKED" in core:
        dataset = "GPQA_CHUNKED"
    elif "GPQA_NOCHUNK" in core:
        dataset = "GPQA_NOCHUNK"
    elif "GPQA" in core:
        dataset = "GPQA"

    elif "IFEval" in core:
        dataset = "IFEval"

    elif "MMLU" in core:
        dataset = "MMLU-Pro"

    elif "Omni-MATH_CHUNKED" in core:
        dataset = "Omni-MATH_CHUNKED"
    elif "Omni-MATH_NOCHUNK" in core:
        dataset = "Omni-MATH_NOCHUNK"
    elif "Omni" in core:
        dataset = "Omni-MATH"
    '''
    
    if "GPQA_CHUNKED" in core:
        dataset = "GPQA_CHUNKED"
    elif "GPQA_NOCHUNK" in core:
        dataset = "GPQA_NOCHUNK"
    elif "GPQA" in core:
        dataset = "GPQA"
    elif "IFEval" in core and "NOCHUNK.jsonl" not in core and "CHUNKED.jsonl" not in core:
        dataset = "IFEval" 
    elif "IFEval" in core and "CHUNKED.jsonl" in core:
        dataset = "IFEval_CHUNKED"
    elif "IFEval" in core and "NOCHUNK.jsonl" in core:
        dataset = "IFEval_NOCHUNK"      

    elif "MMLU" in core:
        dataset = "MMLU-Pro"

    elif "Omni-MATH_CHUNKED" in core:
        dataset = "Omni-MATH_CHUNKED"
    elif "Omni-MATH_NOCHUNK" in core:
        dataset = "Omni-MATH_NOCHUNK"
    elif "Omni" in core:
        dataset = "Omni-MATH"

    ########################################################
    # MODEL
    ########################################################
    model_type = None
    if "nemotron" in core:
        model_type = "nemotron"
    elif "qwen4b" in core:
        model_type = "qwen4b" #qwen4b, qwen8b
    elif "qwen8b" in core:
        model_type = "qwen8b" #qwen4b, qwen8b
    elif "e5" in core:
        model_type = "e5"
    elif "bge" in core:
        model_type = "bge"
    elif "gist" in core:
        model_type = "gist"

    ########################################################
    # MEASURE
    ########################################################
    measure = None
    for m in ["cosine", "correlation", "seuclidean", "euclidean"]:
        if m in core:
            measure = m

    ########################################################
    # THRESHOLD
    ########################################################
    threshold = None
    m_p = re.search(r"(p\d+)", core)
    if m_p:
        threshold = m_p.group(1)

    return {
        "size": size,
        "dataset": dataset,
        "model_type": model_type,
        "measure": measure,
        "threshold": threshold,
        "seed": seed,
        "time_limit": time_limit,
        "core": core
    }


# =========================================================
# Parse solution line: "[ 15 23 44 ]"
# =========================================================
def parse_solution_line(line):
    return list(map(int, re.findall(r"\d+", line)))


# =========================================================
# Read one NuMDS file
# =========================================================
def read_numds_file(path):

    with open(path, "r") as f:
        lines = f.readlines()

    if len(lines) < 2:
        raise ValueError("Invalid NuMDS file format")

    ########################################################
    # First line:
    # graphs_dimacs/...dimacs 10 1
    ########################################################
    header = lines[0].strip().split()

    graph_path = header[0]

    seed_from_file = None
    objective = None

    if len(header) > 1:
        try:
            seed_from_file = int(header[1])
        except:
            pass

    if len(header) > 2:
        try:
            objective = int(header[2])
        except:
            pass

    ########################################################
    # second line: solution
    ########################################################
    solution = parse_solution_line(lines[1])

    return graph_path, seed_from_file, objective, solution


# =========================================================
# Parse ALL NuMDS files in directory
# =========================================================
def numds_parse_all(DIR):

    rows = []

    files = glob.glob(os.path.join(DIR, "*.out"))

    print("Found NuMDS files:", len(files))

    for file in files:

        try:
            #print("Reading:", file)

            meta = parse_numds_filename(file)

            graph_path, seed_file, obj, solution = read_numds_file(file)

            seed = meta["seed"] if meta["seed"] is not None else seed_file

            row = {
                "file": os.path.basename(file),

                "size": meta["size"],
                "dataset": meta["dataset"],
                "model_type": meta["model_type"],
                "measure": meta["measure"],
                "threshold": meta["threshold"],
                "seed": seed,

                "objective": obj,
                "solution_size": len(solution),

                "graph_file": os.path.basename(graph_path),
                "solution": solution
            }

            rows.append(row)

        except Exception as e:
            print("ERROR reading:", file)
            print(e)

    return pd.DataFrame(rows)

In [456]:

numds_data = numds_parse_all("numds")
numds_data = numds_data.drop(columns=["file"])


Found NuMDS files: 2970


In [457]:
numds_data = numds_data[["dataset", "model_type", "size", "seed", "measure", "threshold",   "objective", "solution"]]
numds_data[["objective"]].mean()


objective    3.271044
dtype: float64

In [458]:
### ONLINE_MIS results 

import os
import glob
import re
import pandas as pd


# =========================================================
# Parse filename (same logic as NuMDS)
# =========================================================
def parse_online_mis_filename(filename):

    base = os.path.basename(filename)
    base = base.replace(".out", "")

    ########################################################
    # seed + time
    ########################################################
    seed = None
    time_limit = None

    m_seed = re.search(r"seed(\d+)", base)
    if m_seed:
        seed = int(m_seed.group(1))

    m_time = re.search(r"t(\d+)", base)
    if m_time:
        time_limit = int(m_time.group(1))

    ########################################################
    # remove suffix after .dimacs
    ########################################################
    if ".dimacs" in base:
        core = base.split(".dimacs")[0]
    else:
        core = base

    ########################################################
    # tokens
    ########################################################
    tokens = core.split("_")

    print(tokens)
    ########################################################
    # SIZE
    ########################################################
    size = "unknown"
    if "large" in tokens:
        size = "large"
    elif "small" in tokens[0]:
        size = "small"

    ########################################################
    # DATASET (preserve variants)
    ########################################################
    dataset = None

    if "GPQA_CHUNKED" in core:
        dataset = "GPQA_CHUNKED"
    elif "GPQA_NOCHUNK" in core:
        dataset = "GPQA_NOCHUNK"
    elif "GPQA" in core:
        dataset = "GPQA"
    elif "IFEval" in core and "NOCHUNK.jsonl" not in core and "CHUNKED.jsonl" not in core:
        dataset = "IFEval" 
    elif "IFEval" in core and "CHUNKED.jsonl" in core:
        dataset = "IFEval_CHUNKED"
    elif "IFEval" in core and "NOCHUNK.jsonl" in core:
        dataset = "IFEval_NOCHUNK"      

    elif "MMLU" in core:
        dataset = "MMLU-Pro"

    elif "Omni-MATH_CHUNKED" in core:
        dataset = "Omni-MATH_CHUNKED"
    elif "Omni-MATH_NOCHUNK" in core:
        dataset = "Omni-MATH_NOCHUNK"
    elif "Omni" in core:
        dataset = "Omni-MATH"

    ########################################################
    # MODEL
    ########################################################
    model_type = None
    if "nemotron" in core:
        model_type = "nemotron"
    elif "qwen4b" in core:
        model_type = "qwen4b"
    elif "qwen8b" in core:
        model_type = "qwen8b"
    elif "e5" in core:
        model_type = "e5"
    elif "bge" in core:
        model_type = "bge"
    elif "gist" in core:
        model_type = "gist"

    ########################################################
    # MEASURE
    ########################################################
    measure = None
    for m in ["cosine", "correlation", "seuclidean", "euclidean"]:
        if m in core:
            measure = m

    ########################################################
    # THRESHOLD
    ########################################################
    threshold = None
    m_p = re.search(r"(p\d+)", core)
    if m_p:
        threshold = m_p.group(1)

    return {
        "size": size,
        "dataset": dataset,
        "model_type": model_type,
        "measure": measure,
        "threshold": threshold,
        "seed": seed,
        "time_limit": time_limit
    }


# =========================================================
# Read REDUMIS / Online MIS solution file
# =========================================================
def read_online_mis_file(path):

    with open(path, "r") as f:
        lines = [l.strip() for l in f.readlines() if l.strip() != ""]

    # each line is 0 or 1
    binary_vector = [int(x) for x in lines]

    ########################################################
    # solution = indices where value == 1
    ########################################################
    solution = [i for i, v in enumerate(binary_vector) if v == 1]

    ########################################################
    # objective = number of selected nodes
    ########################################################
    objective = len(solution)

    return solution, objective


# =========================================================
# Parse ALL online MIS files
# =========================================================
def online_mis_parse_all(DIR):

    rows = []

    files = glob.glob(os.path.join(DIR, "*.out"))

    print("Found Online MIS files:", len(files))

    for file in files:

        try:
            #print("Reading:", file)

            meta = parse_online_mis_filename(file)

            solution, obj = read_online_mis_file(file)

            row = {
                "file": os.path.basename(file),

                "size": meta["size"],
                "dataset": meta["dataset"],
                "model_type": meta["model_type"],
                "measure": meta["measure"],
                "threshold": meta["threshold"],
                "seed": meta["seed"],

                "objective": obj,
                "solution_size": len(solution),

                "solution": solution
            }

            rows.append(row)

        except Exception as e:
            print("ERROR reading:", file)
            print(e)

    return pd.DataFrame(rows)



online_mis_data = online_mis_parse_all("online_mis")


Found Online MIS files: 2849
['embeddings', 'large', 'Omni-MATH', 'NOCHUNK.jsonl', 'qwen4b', 'seuclidean', 'p20', 'seed50', 't600', 'redumis']
['small', 'embeds', 'sp', 'MMLU-Pro', 'fixed.jsonl', 'e5', 'cosine', 'p90', 'seed40', 't600', 'redumis']
['embeddings', 'large', 'IFEval', 'CHUNKED.jsonl', 'nemotron', 'correlation', 'p95', 'seed50', 't600', 'redumis']
['embeddings', 'large', 'GPQA', 'CHUNKED.jsonl', 'nemotron', 'cosine', 'p90', 'seed20', 't600', 'redumis']
['embeddings', 'large', 'GPQA', 'NOCHUNK.jsonl', 'qwen4b', 'cosine', 'p20', 'seed30', 't600', 'redumis']
['small', 'embeds', 'sp', 'Omni-MATH', 'fixed.jsonl', 'e5', 'cosine', 'p80', 'seed50', 't600', 'redumis']
['small', 'embeds', 'sp', 'IFEval', 'fixed.jsonl', 'e5', 'cosine', 'p10', 'seed10', 't600', 'redumis']
['small', 'embeds', 'sp', 'GPQA', 'fixed.jsonl', 'bge', 'cosine', 'p90', 'seed40', 't600', 'redumis']
['embeddings', 'large', 'IFEval', 'NOCHUNK.jsonl', 'nemotron', 'seuclidean', 'p90', 'seed20', 't600', 'redumis']
['

In [459]:

online_mis_data = online_mis_data[["dataset", "model_type", "size", "seed", "measure", "threshold",   "objective", "solution"]]
online_mis_data[["objective"]].mean()

objective    312.605476
dtype: float64

In [460]:
## REDUMIS results:

redumis_data = online_mis_parse_all("redumis")

redumis_data = redumis_data[["dataset", "model_type", "size", "seed", "measure", "threshold",   "objective", "solution"]]

Found Online MIS files: 2375
['embeddings', 'large', 'Omni-MATH', 'NOCHUNK.jsonl', 'qwen4b', 'seuclidean', 'p20', 'seed50', 't600', 'redumis']
['small', 'embeds', 'sp', 'MMLU-Pro', 'fixed.jsonl', 'e5', 'cosine', 'p90', 'seed40', 't600', 'redumis']
['embeddings', 'large', 'IFEval', 'CHUNKED.jsonl', 'nemotron', 'correlation', 'p95', 'seed50', 't600', 'redumis']
['embeddings', 'large', 'GPQA', 'CHUNKED.jsonl', 'nemotron', 'cosine', 'p90', 'seed20', 't600', 'redumis']
['embeddings', 'large', 'Omni-MATH', 'NOCHUNK.jsonl', 'nemotron', 'correlation', 'p95', 'seed40', 't600', 'redumis']
['embeddings', 'large', 'GPQA', 'NOCHUNK.jsonl', 'qwen4b', 'cosine', 'p20', 'seed30', 't600', 'redumis']
['small', 'embeds', 'sp', 'Omni-MATH', 'fixed.jsonl', 'e5', 'cosine', 'p80', 'seed50', 't600', 'redumis']
['small', 'embeds', 'sp', 'GPQA', 'fixed.jsonl', 'bge', 'cosine', 'p90', 'seed40', 't600', 'redumis']
['embeddings', 'large', 'IFEval', 'NOCHUNK.jsonl', 'nemotron', 'seuclidean', 'p90', 'seed20', 't600',

In [461]:
redumis_data[["objective"]].mean()  ## REDUMIS vs ONLINE_MIS vs. GREEDY vs. CPLEX (317 vs. 312 vs. 315 vs. 316)
#greedy_data[greedy_data["alg"] == "alg0"]["objective"].mean()

cplex_data[cplex_data["alg"] == "alg0"]["objective"].mean()    ## MIS
cplex_data[cplex_data["alg"] == "alg1"]["objective"].mean()    ## MDS

3.271043771043771

In [462]:
### save results for different solvers: merging CPLEX and GREDY  
DIR = "results/"

ALGS_INDEX = {0: "MIS", 1: "MDS"} 
DATASETS = {  "CPLEX" : cplex_data, "GREEDY" : greedy_data, "NUMDS" : numds_data, "ONLINE_MIS" : online_mis_data, "REDUMIS": redumis_data }

for  p_id, problem_name in ALGS_INDEX.items():
    for alg_name, alg_data in DATASETS.items():
        if alg_name in ["CPLEX", "GREEDY"]:
            filename_out = DIR + problem_name + "_" + alg_name + ".csv"
            data_to_save = alg_data[alg_data["alg"] == "alg" + str(p_id) ] 
            data_to_save = data_to_save.drop(columns=["alg"], axis=1)
            data_to_save.to_csv(filename_out, index=False)
        if alg_name == "NUMDS":
            filename_out = DIR + "MDS_" + "numds" + ".csv"
            alg_data.to_csv(filename_out, index=False)
        if alg_name in ["ONLINE_MIS", "REDUMIS"]:
            filename_out = DIR + "MIS_" +  alg_name + ".csv"
            if alg_name == "REDUMIS":
                alg_data.to_csv(filename_out, index=False)
            else:
                alg_data.to_csv(filename_out, index=False) #alg_data["seed"] > 10



In [463]:
import pandas as pd
import numpy as np

############################################################
# REQUIRED CPLEX COLUMNS (force existence)
############################################################

CPLEX_COLS = [
    "cplex_objective",
    "cplex_time",
    "cplex_LB",
    "cplex_solution_size"
]

GREEDY_COLS = [
    "greedy_objective",
    "greedy_time",
    "greedy_solution_size"
]


############################################################
# Safe rename + column normalization
############################################################

def normalize_cplex(df):
    df = df.copy()

    df = df.rename(columns={
        "objective": "cplex_objective",
        "time": "cplex_time",
        "LB": "cplex_LB",
        "solution_size": "cplex_solution_size"
    })

    # 🔥 FORCE missing columns to exist
    for col in CPLEX_COLS:
        if col not in df.columns:
            df[col] = np.nan

    return df


def normalize_greedy(df):
    df = df.copy()

    df = df.rename(columns={
        "objective": "greedy_objective",
        "time": "greedy_time",
        "solution_size": "greedy_solution_size"
    })

    for col in GREEDY_COLS:
        if col not in df.columns:
            df[col] = np.nan

    return df


############################################################
# Apply normalization
############################################################

cplex_mis = normalize_cplex(cplex_mis)
cplex_mds = normalize_cplex(cplex_mds)

greedy_mis = normalize_greedy(greedy_mis)
greedy_mds = normalize_greedy(greedy_mds)

############################################################
# Split algorithms
############################################################

cplex_mis = normalize_cplex(cplex_data[cplex_data["alg"] == "alg0"].drop(columns=["alg"]))
cplex_mds = normalize_cplex(cplex_data[cplex_data["alg"] == "alg1"].drop(columns=["alg"]))

greedy_mis = normalize_greedy(greedy_data[greedy_data["alg"] == "alg0"].drop(columns=["alg"]))
greedy_mds = normalize_greedy(greedy_data[greedy_data["alg"] == "alg1"].drop(columns=["alg"]))


############################################################
# Ensure numeric stability
############################################################

for df in [cplex_mis, cplex_mds]:
    for col in CPLEX_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for df in [greedy_mis, greedy_mds]:
    for col in GREEDY_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

merge_keys = [
    "dataset",
    "model_type",
    "size",
    "measure",
    "threshold"
]

############################################################
# MERGE MIS
############################################################

merged_mis = pd.merge(
    greedy_mis,
    cplex_mis,
    on=merge_keys,
    how="left"
)


############################################################
# MERGE MDS
############################################################

merged_mds = pd.merge(
    greedy_mds,
    cplex_mds,
    on=merge_keys,
    how="left"
)


############################################################
# Final cleanup (optional but safe)
############################################################

for col in CPLEX_COLS:
    if col in merged_mis.columns:
        merged_mis[col] = merged_mis[col].replace({None: np.nan})

    if col in merged_mds.columns:
        merged_mds[col] = merged_mds[col].replace({None: np.nan})



In [464]:
### save into deterministic_mis.csv and deterministic_mds.csv

merged_mds_to_save = merged_mds[["dataset", "model_type", "size", "measure", "threshold", "greedy_objective", "solution_x", "cplex_objective", "solution_y" ]]
merged_mis_to_save = merged_mis[["dataset", "model_type", "size", "measure", "threshold", "greedy_objective", "solution_x", "cplex_objective", "solution_y" ]]

merged_mds_to_save_renamed = merged_mds_to_save.rename(columns={ "solution_x" : "greedy_solution", "solution_y": "cplex_solution" })
merged_mds_to_save_renamed

merged_mis_to_save_renamed = merged_mis_to_save.rename(columns={ "solution_x" : "greedy_solution", "solution_y": "cplex_solution" })
merged_mis_to_save_renamed

## save file:
merged_mis_to_save_renamed.to_csv("results_summary/deterministic_mis.csv", index=False)
merged_mds_to_save_renamed.to_csv("results_summary/deterministic_mds.csv", index=False)




In [465]:
import pandas as pd
import numpy as np

############################################################
# Merge keys
############################################################

merge_keys = [
    "dataset",
    "model_type",
    "size",
    "seed",
    "measure",
    "threshold"
]


############################################################
# Normalize ONLINE-MIS dataframe
############################################################

def normalize_online(df):

    df = df.copy()

    ########################################################
    # Rename columns
    ########################################################
    df = df.rename(columns={
        "objective": "online_objective",
        "time": "online_time",
        "solution_size": "online_solution_size",
        "solution": "online_solution"
    })

    ########################################################
    # Ensure columns exist
    ########################################################
    required_cols = [
        "online_objective",
        "online_time",
        "online_solution_size",
        "online_solution"
    ]

    for col in required_cols:

        if col not in df.columns:
            df[col] = np.nan

    return df


############################################################
# Normalize REDUMIS dataframe
############################################################

def normalize_redumis(df):

    df = df.copy()

    ########################################################
    # Rename columns
    ########################################################
    df = df.rename(columns={
        "objective": "redumis_objective",
        "time": "redumis_time",
        "solution_size": "redumis_solution_size",
        "solution": "redumis_solution"
    })

    ########################################################
    # Ensure columns exist
    ########################################################
    required_cols = [
        "redumis_objective",
        "redumis_time",
        "redumis_solution_size",
        "redumis_solution"
    ]

    for col in required_cols:

        if col not in df.columns:
            df[col] = np.nan

    return df


############################################################
# Apply normalization
############################################################

online_df = normalize_online(online_mis_data)

redumis_df = normalize_redumis(redumis_data)


############################################################
# Convert numeric columns safely
############################################################

online_numeric = [
    "online_objective",
    "online_time",
    "online_solution_size"
]

redumis_numeric = [
    "redumis_objective",
    "redumis_time",
    "redumis_solution_size"
]


for col in online_numeric:

    online_df[col] = pd.to_numeric(
        online_df[col],
        errors="coerce"
    )


for col in redumis_numeric:

    redumis_df[col] = pd.to_numeric(
        redumis_df[col],
        errors="coerce"
    )


############################################################
# DEBUG DUPLICATES
############################################################

print("===================================")
print("ONLINE DUPLICATES")
print("===================================")

online_duplicates = online_df[
    online_df.duplicated(
        subset=merge_keys,
        keep=False
    )
]

print("Duplicated ONLINE rows:",
      len(online_duplicates))

print(
    online_duplicates[merge_keys]
    .sort_values(merge_keys)
)



print("===================================")
print("REDUMIS DUPLICATES")
print("===================================")

redumis_duplicates = redumis_df[
    redumis_df.duplicated(
        subset=merge_keys,
        keep=False
    )
]

print("Duplicated REDUMIS rows:",
      len(redumis_duplicates))

print(
    redumis_duplicates[merge_keys]
    .sort_values(merge_keys)
)


############################################################
# REMOVE DUPLICATES
############################################################

online_df = online_df.drop_duplicates(
    subset=merge_keys
)

redumis_df = redumis_df.drop_duplicates(
    subset=merge_keys
)



############################################################
# FINAL SAFE MERGE
############################################################

merged_mh_mis = pd.merge(
    online_df,
    redumis_df,
    on=merge_keys,
    how="inner",               # ONLY matching rows
    validate="one_to_one"      # ensures no explosion
)

#merged_mh_mis
#online_df

ONLINE DUPLICATES
Duplicated ONLINE rows: 0
Empty DataFrame
Columns: [dataset, model_type, size, seed, measure, threshold]
Index: []
REDUMIS DUPLICATES
Duplicated REDUMIS rows: 0
Empty DataFrame
Columns: [dataset, model_type, size, seed, measure, threshold]
Index: []


In [466]:
#online_df[online_df["dataset"] =="IFEval_CHUNKED"]
#redumis_data[ (redumis_data["dataset"] == "IFEval_fixed")] #& (redumis_data["model_type"] == "nemotron" )  & (redumis_data["size"] == "large" )   & ( redumis_data["measure"] == "correlation" ) & ( redumis_data["threshold"] == "p10" ) ]  # IFEval   nemotron  large    20  correlation       p10

In [471]:
merged_mh_mis_save = merged_mh_mis[["dataset", "model_type", "size", "seed", "measure", "threshold", "online_objective", "online_solution", "redumis_objective", "redumis_solution"]]
merged_mh_mis_save.to_csv("results_summary/mh_mis_results.csv", index=False)

In [469]:
redumis_data[redumis_data["dataset"]=="IFEval"]

,dataset,model_type,size,seed,measure,threshold,objective,solution
16,IFEval,bge,small,50,cosine,p90,335,"[0, 1, 2, 3, 5, 6, 7, 9, 12, 14, 17, 20, 22, 2..."
19,IFEval,e5,small,40,euclidean,p50,107,"[7, 22, 23, 30, 33, 40, 46, 51, 52, 59, 64, 77..."
23,IFEval,gist,small,30,euclidean,p50,103,"[3, 6, 7, 12, 22, 23, 30, 32, 43, 46, 49, 53, ..."
29,IFEval,bge,small,40,cosine,p10,27,"[43, 61, 66, 123, 130, 136, 137, 192, 230, 253..."
32,IFEval,gist,small,50,euclidean,p95,382,"[0, 1, 3, 5, 6, 7, 8, 9, 12, 13, 14, 17, 20, 2..."
...,...,...,...,...,...,...,...,...
2327,IFEval,bge,small,40,correlation,p20,45,"[5, 7, 9, 22, 33, 59, 64, 77, 88, 110, 113, 12..."
2336,IFEval,e5,small,50,cosine,p90,354,"[0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 13, 14, 17, 19..."
2337,IFEval,bge,small,20,cosine,p80,245,"[0, 1, 3, 5, 6, 7, 9, 12, 14, 17, 20, 22, 23, ..."
2360,IFEval,bge,small,30,euclidean,p20,46,"[5, 6, 7, 9, 33, 59, 64, 77, 88, 110, 113, 123..."
